# Logical Reasoning for Planning
This notebook is the solution for the lab `Logic.pdf` (constructing and testing a simple planning agent with an LLM), including the optional Prolog extension.

**LLM use:** the planner in Task 2 was generated with Claude (Anthropic) from the prompt shown there, then modified and tested. The Prolog program follows the lab sheet. Changes are listed in the reflection.

### Task 0: Understand the planning problem

**(a) Initial state:** $I = \{At(Robot, A),\ At(Package, A)\}$

**(b) Goal:** $G = \{At(Package, C)\}$

**(c) Actions** (locations connected as $A - B - C$):
- $Move(A,B)$, $Move(B,A)$, $Move(B,C)$, $Move(C,B)$
- $PickUp(Package, l)$ for $l \in \{A, B, C\}$
- $Drop(Package, l)$ for $l \in \{A, B, C\}$

**(d) Preconditions and effects**

| Action | Preconditions | Add effects | Delete effects |
|---|---|---|---|
| $Move(x, y)$, $x,y$ connected | $At(Robot, x)$ | $At(Robot, y)$ | $At(Robot, x)$ |
| $PickUp(Package, l)$ | $At(Robot, l)$, $At(Package, l)$, $\neg Holding(Package)$ | $Holding(Package)$ | $At(Package, l)$ |
| $Drop(Package, l)$ | $At(Robot, l)$, $Holding(Package)$ | $At(Package, l)$ | $Holding(Package)$ |

The negative precondition $\neg Holding(Package)$ on $PickUp$ is my addition; it is redundant here (the package cannot be both held and at $l$) but makes the intent explicit. While the robot holds the package, the package's location is implicitly the robot's location.

**Initially applicable actions**

- $PickUp(Package, A)$: **applicable**. Its preconditions $At(Robot, A)$ and $At(Package, A)$ are both in $I$ (and $Holding(Package) \notin I$), so $I \models Preconditions(PickUp(Package, A))$.
- $Drop(Package, C)$: **not applicable**. It needs $At(Robot, C)$ and $Holding(Package)$; neither is in $I$.
- The only other applicable action in $I$ is $Move(A, B)$.

### Task 1: Construct a plan by hand

**Note on the lab sheet's example:** the sheet suggests reasoning about $Move(A,B)$, $PickUp(Package, B)$, $Move(B,C)$, $Drop(Package, C)$. That sequence is **not valid**: after $Move(A,B)$ the package is still at $A$, so $PickUp(Package, B)$ fails its precondition $At(Package, B)$. The robot must pick up first.

Valid plan: $PickUp(Package, A),\ Move(A, B),\ Move(B, C),\ Drop(Package, C)$

| State | After action | Facts |
|---|---|---|
| $S_0$ | – | $At(Robot, A),\ At(Package, A)$ |
| $S_1$ | $PickUp(Package, A)$ | $At(Robot, A),\ Holding(Package)$ |
| $S_2$ | $Move(A, B)$ | $At(Robot, B),\ Holding(Package)$ |
| $S_3$ | $Move(B, C)$ | $At(Robot, C),\ Holding(Package)$ |
| $S_4$ | $Drop(Package, C)$ | $At(Robot, C),\ At(Package, C)$ |

$S_4 \models G$, so the plan is valid. It has 4 actions, which is minimal: the robot must pick up, make two moves ($A \to C$ has no direct link), and drop.

### Task 2: Ask an LLM to implement the planner

**Prompt used** (the lab's specification, plus the domain):

> I want to implement a simple planning agent in Python.
> Represent a state as a set of logical propositions.
> Each action should contain: a name; positive preconditions; negative preconditions; positive effects; negative effects.
> An action is applicable if all of its preconditions are satisfied by the current state.
> When an action is applied: 1. remove its negative effects from the state; 2. add its positive effects to the state.
> Use breadth-first search to find a sequence of actions that achieves a specified goal.
> The program should also: detect when no plan exists; print the resulting sequence of actions; print the states reached after each action.
> Explain the implementation and identify any assumptions you make.
> Domain: a robot and a package, locations A, B, C connected A–B and B–C. Actions Move(x, y), PickUp(Package, l), Drop(Package, l) with the preconditions and effects in the table I give you. Initial state {At(Robot, A), At(Package, A)}, goal {At(Package, C)}.

**Assumptions the LLM stated (and I accepted):** closed-world assumption (any proposition not in the state set is false); the goal is a conjunction of positive propositions; actions are fully grounded (no variables at search time); deterministic actions.

In [1]:
from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, List, Optional


@dataclass(frozen=True)
class Action:
    name: str
    pos_prec: FrozenSet[str] = field(default_factory=frozenset)
    neg_prec: FrozenSet[str] = field(default_factory=frozenset)
    add: FrozenSet[str] = field(default_factory=frozenset)      # positive effects
    delete: FrozenSet[str] = field(default_factory=frozenset)   # negative effects

    def applicable(self, state: FrozenSet[str]) -> bool:
        # LOGIC: S |= Preconditions(a)   (closed-world: absent means false)
        return self.pos_prec <= state and not (self.neg_prec & state)

    def apply(self, state: FrozenSet[str]) -> FrozenSet[str]:
        # EFFECTS: remove negative effects, then add positive effects
        return (state - self.delete) | self.add

    def __repr__(self):
        return self.name


def satisfies(state, goal) -> bool:
    return goal <= state                    # GOAL TEST: S |= G


def bfs_plan(initial, goal, actions) -> Optional[List[Action]]:
    """SEARCH: breadth-first search over states; returns the shortest plan or None."""
    start = frozenset(initial)
    goal = frozenset(goal)
    frontier = deque([start])
    parent = {start: None}                  # state -> (previous state, action)
    while frontier:
        state = frontier.popleft()
        if satisfies(state, goal):
            plan = []
            while parent[state] is not None:
                state, a = parent[state]
                plan.append(a)
            return plan[::-1]
        for a in actions:
            if a.applicable(state):         # logic decides what is possible ...
                nxt = a.apply(state)
                if nxt not in parent:       # ... search decides what to try next
                    parent[nxt] = (state, a)
                    frontier.append(nxt)
    return None


def show(state):
    return "{" + ", ".join(sorted(state)) + "}"


def run_plan(initial, goal, actions, title=""):
    print(f"=== {title}")
    plan = bfs_plan(initial, goal, actions)
    if plan is None:
        print("No plan found\n")
        return None
    state = frozenset(initial)
    print(f"S0: {show(state)}")
    for i, a in enumerate(plan, 1):
        state = a.apply(state)
        print(f"S{i}: {show(state):<45} <- {a}")
    print()
    return plan


# ---- warehouse domain --------------------------------------------------------
LOCS = ["A", "B", "C"]
CONNECTED = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]

def move(x, y):
    return Action(f"Move({x}, {y})", pos_prec=frozenset({f"At(Robot, {x})"}),
                  add=frozenset({f"At(Robot, {y})"}), delete=frozenset({f"At(Robot, {x})"}))

def pickup(l):
    return Action(f"PickUp(Package, {l})",
                  pos_prec=frozenset({f"At(Robot, {l})", f"At(Package, {l})"}),
                  neg_prec=frozenset({"Holding(Package)"}),
                  add=frozenset({"Holding(Package)"}), delete=frozenset({f"At(Package, {l})"}))

def drop(l):
    return Action(f"Drop(Package, {l})",
                  pos_prec=frozenset({f"At(Robot, {l})", "Holding(Package)"}),
                  add=frozenset({f"At(Package, {l})"}), delete=frozenset({"Holding(Package)"}))

MOVES = [move(x, y) for x, y in CONNECTED]
PICKUPS = [pickup(l) for l in LOCS]
DROPS = [drop(l) for l in LOCS]
ACTIONS = MOVES + PICKUPS + DROPS

I = {"At(Robot, A)", "At(Package, A)"}
G = {"At(Package, C)"}

plan = run_plan(I, G, ACTIONS, "Warehouse problem")

=== Warehouse problem
S0: {At(Package, A), At(Robot, A)}
S1: {At(Robot, A), Holding(Package)}              <- PickUp(Package, A)
S2: {At(Robot, B), Holding(Package)}              <- Move(A, B)
S3: {At(Robot, C), Holding(Package)}              <- Move(B, C)
S4: {At(Package, C), At(Robot, C)}                <- Drop(Package, C)



**Think about it – where the specification appears in the code:**

| Idea | Where |
|---|---|
| Preconditions → when is an action applicable? | `Action.applicable` – positive preconditions must be a subset of the state, negative ones must be absent |
| Effects → how does the state change? | `Action.apply` – `(state - delete) | add` |
| Goal → when does planning terminate? | `satisfies(state, goal)` checked on each popped state; or the frontier empties → `None` |
| BFS → how are alternatives explored? | `deque` FIFO frontier in `bfs_plan`, with `parent` as the visited set; shallowest plans are found first |

### Task 3: Test the generated planner

An **independent validator** is used for every returned plan: it re-executes the plan from scratch and checks each action's preconditions, without using any of the search code.

In [2]:
def validate(initial, goal, plan):
    """Independent check: re-execute the plan, verifying every precondition."""
    state = frozenset(initial)
    for i, a in enumerate(plan):
        missing = a.pos_prec - state
        violated = a.neg_prec & state
        if missing or violated:
            return False, f"step {i+1} {a}: missing {set(missing) or '{}'}, violated {set(violated) or '{}'}"
        state = a.apply(state)
    return satisfies(state, frozenset(goal)), f"final state {show(state)}"

results = []

# ---- Test A: solvable problem ----
planA = run_plan(I, G, ACTIONS, "Test A: original warehouse")
okA = validate(I, G, planA)
assert okA[0] and len(planA) == 4
results.append(("A", I, G, planA, okA))

# ---- Test B: impossible problem (no PickUp actions) ----
planB = run_plan(I, G, MOVES + DROPS, "Test B: PickUp removed")
assert planB is None
results.append(("B", I, G, planB, (True, "correctly reports no plan")))

# ---- Test C1: irrelevant actions added ----
# Extra robot-only actions that never move the package: a direct shortcut A->C and a no-op 'Beep'.
shortcut = move("A", "C")
beep = Action("Beep", pos_prec=frozenset())     # always applicable, no effect
planC1 = run_plan(I, G, ACTIONS + [shortcut, beep], "Test C1: irrelevant / extra robot actions")
okC1 = validate(I, G, planC1)
assert okC1[0] and "Beep" not in map(str, planC1)
results.append(("C1", I, G, planC1, okC1))

# ---- Test C2: robot CAN reach C, package can't move ----
# Only robot movements: the robot reaches C easily, but At(Robot, C) must NOT count as the goal.
planC2 = run_plan(I, G, MOVES + [shortcut], "Test C2: robot can reach C, package cannot")
assert planC2 is None
results.append(("C2", I, G, planC2, (True, "correctly reports no plan")))

print("All tests passed")

=== Test A: original warehouse
S0: {At(Package, A), At(Robot, A)}
S1: {At(Robot, A), Holding(Package)}              <- PickUp(Package, A)
S2: {At(Robot, B), Holding(Package)}              <- Move(A, B)
S3: {At(Robot, C), Holding(Package)}              <- Move(B, C)
S4: {At(Package, C), At(Robot, C)}                <- Drop(Package, C)

=== Test B: PickUp removed
No plan found

=== Test C1: irrelevant / extra robot actions
S0: {At(Package, A), At(Robot, A)}
S1: {At(Robot, A), Holding(Package)}              <- PickUp(Package, A)
S2: {At(Robot, C), Holding(Package)}              <- Move(A, C)
S3: {At(Package, C), At(Robot, C)}                <- Drop(Package, C)

=== Test C2: robot can reach C, package cannot
No plan found

All tests passed


In [3]:
# What would go wrong with a buggy goal test that checks the ROBOT instead of the package?
buggy_goal = {"At(Robot, C)"}
bad = bfs_plan(I, buggy_goal, MOVES)
print("Buggy goal 'At(Robot, C)' gives plan:", bad)
print("Validator against the REAL goal:", validate(I, G, bad))

Buggy goal 'At(Robot, C)' gives plan: [Move(A, B), Move(B, C)]
Validator against the REAL goal: (False, 'final state {At(Package, A), At(Robot, C)}')


**Recorded results**

| Test | Initial state | Goal | Plan found? | Plan | Actually valid? |
|---|---|---|---|---|---|
| A | $\{At(Robot,A), At(Package,A)\}$ | $At(Package,C)$ | Yes | PickUp(Package, A), Move(A, B), Move(B, C), Drop(Package, C) | Yes – every precondition holds, final state satisfies $G$ |
| B | same | $At(Package,C)$ | No | "No plan found" | Correct – no action can ever change the package's location |
| C1 | same (+ `Move(A,C)` shortcut, `Beep`) | $At(Package,C)$ | Yes | PickUp(Package, A), Move(A, C), Drop(Package, C) | Yes – shorter plan using the extra move; `Beep` is never used |
| C2 | same (only robot moves) | $At(Package,C)$ | No | "No plan found" | Correct – the robot reaches C but the goal is not satisfied |

The extra cell shows the failure mode Test C targets: if the goal were mistakenly written as $At(Robot, C)$, the planner happily returns `[Move(A, B), Move(B, C)]` – the robot arrives at C while the package is still at A – which the validator rejects for the real goal.

### Task 4: Logic and search

**Completed pipeline:**

Current state → Check action preconditions → **Select the applicable actions** → Generate successor state → Search over alternatives → Goal?

- **Logical reasoning** is used in `applicable` (deciding $S \models Preconditions(a)$), in `apply` (computing the logical consequences of an action under the STRIPS/closed-world assumption) and in the goal test ($S \models G$).
- **Search** is used in `bfs_plan`: it decides *in which order* the applicable actions and resulting states are explored, keeps track of states already reached, and returns the first (shortest) action sequence whose final state satisfies the goal.

In my own words: logic acts as a filter and simulator – for any state it says exactly which moves are legal and what the world looks like afterwards. On its own it cannot choose between the legal moves. Search uses that filtered, simulated state space as a graph and systematically tries sequences until one reaches a goal state. "Logic determines what is possible; search determines what to try." This is the same BFS as in the Search lab; only the state changed from a grid cell to a set of propositions, and the successor function changed from "neighbouring free cells" to "states reachable by an applicable action".

### Task 5 (optional): Can the LLM verify its own plan?

I asked the LLM: *"For every action in the plan, identify its preconditions and show that those preconditions are satisfied in the state in which the action is executed."*

Its explanation for the 4-step plan walked through the same state table as Task 1 and was correct. To see whether an explanation alone can be relied on, the cell below also executes the sequence suggested on the lab sheet, which looks just as plausible when described in words.

In [4]:
# The lab sheet's example sequence, executed and checked independently
by_name = {str(a): a for a in ACTIONS}
sheet_example = [by_name[n] for n in ["Move(A, B)", "PickUp(Package, B)", "Move(B, C)", "Drop(Package, C)"]]
print("Lab-sheet example :", validate(I, G, sheet_example))
print("Planner's plan    :", validate(I, G, plan))

Lab-sheet example : (False, "step 2 PickUp(Package, B): missing {'At(Package, B)'}, violated {}")
Planner's plan    : (True, 'final state {At(Package, C), At(Robot, C)}')


**Which should you trust more?** (b), the independently executed state transitions. The LLM's explanation is text produced to *sound like* a justification; it can assert that a precondition holds without tracking the state (a fluent justification can be produced for a sequence whether or not it is valid). The executed transitions are a mechanical check of exactly the definitions in the specification, and they flag the lab sheet's sequence as invalid at step 2. A generated explanation is not the same as an independent verification.

### Reflection questions

1. **Why specify preconditions and effects first?** They *are* the problem. If they are not fixed before prompting, the LLM fills the gaps with its own guesses (e.g. allowing a direct A→C move, or letting the robot pick up a package anywhere), and the resulting planner solves a different problem. A written specification is also what the generated code is tested against.
2. **Error if preconditions weren't checked:** the planner could return $[Drop(Package, C)]$ as a one-step plan from the initial state – "teleporting" a package the robot never picked up to a location the robot never visited.
3. **Why a plan that "looks reasonable" may be invalid:** validity depends on the exact state at each step, which is easy to misjudge by eye. The lab sheet's own example ($Move(A,B)$ then $PickUp(Package, B)$) looks like a natural sequence but fails because the package was left at A.
4. **What the LLM contributed:** the dataclass-based action representation, the BFS implementation with plan reconstruction, the printing of intermediate states, and a list of its assumptions (closed world, grounded actions).
5. **What I verified independently:** the hand-made plan and state table; every returned plan with a separate validator; that impossible problems return "No plan found"; that reaching C with the robot alone is not treated as the goal; and that BFS returns a minimal-length plan.
6. **Where logical reasoning is used:** checking $S \models Preconditions(a)$, computing successor states from add/delete effects (with the closed-world assumption), and the goal test $S \models G$. In the Prolog part it is used again to check proposed moves against the knowledge base.
7. **Relation to search algorithms:** planning is state-space search where states are sets of propositions and the successor function comes from the action schemas. The same BFS (or A* with a heuristic such as the number of unsatisfied goal propositions) applies unchanged.

**LLM-generated vs. modified:** the `Action` class, `bfs_plan` and `run_plan` were generated; I added the negative precondition on `PickUp`, the `validate` function, Tests C1/C2 and the buggy-goal demonstration.

## Optional extension: Prolog as a logical verifier

The cell below writes `planner.pl` and runs the queries with SWI-Prolog (`swipl`). If `swipl` is not installed, the same file can be loaded in [SWISH](https://swish.swi-prolog.org/).

In [5]:
import shutil
import subprocess

PLANNER_PL = """\
% Warehouse connectivity
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

can_move(X,Y) :- connected(X,Y).
valid_move(X,Y) :- connected(X,Y).

% Task 8: chain of rules
wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
"""
with open("planner.pl", "w") as f:
    f.write(PLANNER_PL)


def query(q):
    """Ask SWI-Prolog whether goal q succeeds against planner.pl."""
    if shutil.which("swipl") is None:
        return "swipl not installed - run in SWISH"
    goal = f"({q} -> writeln(true) ; writeln(false))"
    out = subprocess.run(["swipl", "-q", "-g", goal, "-t", "halt", "planner.pl"],
                         capture_output=True, text=True)
    return out.stdout.strip()


print("Task 6")
for q in ["can_move(a,b)", "can_move(a,c)"]:
    print(f"?- {q}.  ->  {query(q)}")

print("\nTask 7")
for q in ["valid_move(a,b)", "valid_move(b,c)", "valid_move(a,c)"]:
    print(f"?- {q}.  ->  {query(q)}")

print("\nTask 7 challenge: Python planner proposes Move(a, c)")
print(f"?- valid_move(a,c).  ->  {query('valid_move(a,c)')}")

print("\nTask 8")
print(f"?- reduce_speed.  ->  {query('reduce_speed')}")

Task 6
?- can_move(a,b).  ->  true
?- can_move(a,c).  ->  false

Task 7
?- valid_move(a,b).  ->  true
?- valid_move(b,c).  ->  true
?- valid_move(a,c).  ->  false

Task 7 challenge: Python planner proposes Move(a, c)
?- valid_move(a,c).  ->  false

Task 8
?- reduce_speed.  ->  true


In [6]:
# Use Prolog as an independent verifier for every Move in the plans produced by the Python planner
import re

def verify_moves_with_prolog(plan):
    for a in plan:
        m = re.fullmatch(r"Move\((\w), (\w)\)", str(a))
        if m:
            x, y = m.group(1).lower(), m.group(2).lower()
            print(f"{str(a):<14} valid_move({x},{y}) -> {query(f'valid_move({x},{y})')}")

print("Plan from Test A:")
verify_moves_with_prolog(planA)
print("\nPlan from Test C1 (domain had an extra A->C shortcut):")
verify_moves_with_prolog(planC1)

Plan from Test A:
Move(A, B)     valid_move(a,b) -> true
Move(B, C)     valid_move(b,c) -> true

Plan from Test C1 (domain had an extra A->C shortcut):
Move(A, C)     valid_move(a,c) -> false


### Task 6: answers

| Query | Result |
|---|---|
| `?- can_move(a,b).` | `true` |
| `?- can_move(a,c).` | `false` |

(a) `can_move(a,b)` is true because the rule `can_move(X,Y) :- connected(X,Y).` unifies `X=a, Y=b`, and the body `connected(a,b)` matches a fact in the knowledge base.

(b) `can_move(a,c)` is not established because there is no fact `connected(a,c)` and no other rule that can derive it. Prolog uses negation as failure / closed-world reasoning: what cannot be proved is reported `false`. Note the rule only covers *one* move; there is no transitive "reachable" rule, so the two-step route via `b` is not considered.

(c) The rule is the Horn clause $\forall X, Y\ (Connected(X, Y) \rightarrow CanMove(X, Y))$; `:-` is read as "if", with the head on the left and the body on the right.

### Task 7: answers

`valid_move(a,b)` → `true`, `valid_move(b,c)` → `true`, `valid_move(a,c)` → `false`, as expected.

**Challenge:** the proposed action $Move(a, c)$ is **not supported** by the warehouse knowledge – `valid_move(a,c)` fails. The last cell shows this catching something real: the plan from Test C1 (where I deliberately added a shortcut action to the Python domain) uses `Move(A, C)`, and the Prolog verifier, which uses only the original warehouse facts, rejects it. The Python program generated the candidate; Prolog checked it independently.

### Task 8: answer

`?- reduce_speed.` succeeds. Prolog tries to prove `reduce_speed`; the rule says it holds if `slippery` holds; `slippery` holds if `wet_road` holds; and `wet_road` is a fact. So the proof chain is:

$$WetRoad \;\Rightarrow\; (WetRoad \rightarrow Slippery) \;\Rightarrow\; (Slippery \rightarrow ReduceSpeed) \;\Rightarrow\; ReduceSpeed$$

(Fact ⇒ Rule ⇒ Rule ⇒ Conclusion, i.e. two applications of modus ponens. Prolog actually finds this by backward chaining from the query.)

### Prolog reflection

1. **Fact vs rule:** a fact is unconditionally true (`connected(a,b).`); a rule states that its head is true *if* its body can be proved (`can_move(X,Y) :- connected(X,Y).`).
2. **Query as entailment:** `?- q.` asks whether `q` can be derived from the facts and rules, i.e. whether $KB \models q$ – with the caveat that Prolog answers `false` for anything it cannot prove (closed world), which is not the same as $KB \models \neg q$.
3. **Why verify a Python plan with Prolog:** the check uses a separate, declarative description of the world. If the Python domain has a bug (like the shortcut in Test C1) the plan can still be "valid" inside Python; an independent knowledge base catches that mismatch.
4. **Advantage when the plan came from an LLM:** the LLM-written code and the LLM's own explanation can share the same mistake. An independent verifier does not inherit that mistake, so agreement between the two is real evidence, rather than the generator grading its own work.